# 19. 전체 자료 저장 범위 정하기

현재 단계는 **M4 벡터 저장을 전체 자료로 확대하기 전 조사**입니다. 저장된 표본 검색은 18번에서 확인했고, 이번에는 전체 자료 중 무엇을 저장하고 무엇을 더 검토할지 살펴봅니다.

- [프로젝트 적용] 기존 PDF 전처리·청킹 결과를 그대로 사용합니다. 청킹 규칙과 모델을 바꾸지 않습니다.
- [프로젝트 추가] 개인 DB 표본과 현재 PDF·모델 버전을 비교하고, 자동 초안·그림 연결·설명 누락 개수를 집계합니다.
- 임베딩 모델 실행, DB 쓰기, 파일 업로드는 없습니다. 토큰을 세는 로컬 도구는 사용합니다.
- 전체 PDF를 읽으므로 준비에 시간이 걸릴 수 있습니다. 커널을 중복 실행하지 말고 진행 표시가 끝날 때까지 기다리세요.
- 새 노트북의 실행 결과는 아직 저장하지 않았습니다.


In [ ]:
from pathlib import Path
import sys

# notebooks에서 실행해도 프로젝트의 src 폴더를 찾아 개인 Python 파일을 불러옵니다.
project_folder = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                       if (folder / "src/car_search_rag/zzong_santafe_lag").is_dir()), None)
if project_folder is None:
    raise FileNotFoundError("프로젝트 폴더에서 이 노트북을 열어 주세요.")
sys.path.insert(0, str(project_folder / "src"))

from car_search_rag.zzong_santafe_lag.storage_plan import build_storage_plan, print_storage_plan


## 1. 전체 자료와 검토 상태 집계

`build_storage_plan()`은 PDF와 개인 DB를 읽고 **조사 결과 딕셔너리**를 돌려줍니다. 딕셔너리는 이름과 값을 짝지어 보관하는 자료입니다. DB 저장 가능한 행이나 벡터를 새로 만드는 함수는 아닙니다.

부모는 주제 전체 글, 청크는 검색할 작은 조각입니다. 자동 초안은 프로그램이 나눈 글이며 원본을 사람이 모두 대조한 자료가 아닙니다.


In [ ]:
# [프로젝트 추가] 이 셀에서 전체 PDF를 한 번 준비합니다. 결과를 plan에 보관합니다.
plan = build_storage_plan(progress=print)
print_storage_plan(plan, limit=5)

# 아래 셀들은 이미 만든 plan을 읽으므로 전체 PDF를 다시 읽지 않습니다.
print("검토 상태별 부모 수:", plan["summary"]["verification_status"])
print("모델:", plan["model_name"])
print("새 임베딩 계산 여부:", plan["embedding_computed"])
print("DB 쓰기·Storage 업로드 여부:", plan["db_written"], plan["storage_uploaded"])


## 2. 그림 개수 읽는 방법

- **PDF 그림 요소:** PDF에서 읽은 페이지별 내부 이미지 키 개수입니다.
- **연결된 고유 그림:** 부모 글에서 참조하는 서로 다른 그림입니다.
- **주제와 그림 연결:** 같은 그림을 두 주제에서 쓰면 연결은 두 개입니다.
- **설명 없는 연결:** 그림이 연결돼 있지만 설명을 아직 작성하지 않은 경우입니다.

본문에 연결되지 않은 요소를 불필요한 그림으로 단정하지 않습니다. 표지·목차·색인 등 제외한 페이지에 있는지 원본 확인이 필요합니다. 전체 그림을 모두 꺼내지 않으므로 전체 Storage 파일 크기는 아직 모릅니다.


In [ ]:
# [프로젝트 추가] 그림 파일과 그림 설명의 완료 상태를 따로 확인합니다.
images = plan["images"]
print("PDF 그림 요소:", images["pdf_image_elements"])
print("연결된 고유 그림:", images["linked_unique_images"])
print("주제와 그림 연결:", images["parent_image_links"])
print("설명 있는 연결:", images["links_with_description"])
print("설명 없는 연결:", images["links_without_description"])
print("본문에 연결되지 않은 PDF 그림 요소:", images["unlinked_pdf_image_elements"])
print("기존 업로드 표본:", images["existing_uploaded_sample_images"])
print("전체 Storage 크기:", images["full_storage_bytes"], "(None = 아직 측정하지 않음)")


## 3. 먼저 확인할 자동 초안

표·안내 목록, 깨진 기호, 경고 경계, 긴 주제를 먼저 확인하도록 간단한 규칙으로 순서를 정했습니다. 오류라고 확정한 목록이나 검색 점수 순위는 아닙니다. 원본 화면과 대조한 뒤에만 검토 상태를 바꿉니다.

모든 자동 초안 목록은 `plan["review_items"]`에 있습니다.


In [ ]:
# [프로젝트 추가] 처음에는 다섯 주제만 살펴봅니다. PDF 쪽수는 뷰어 기준입니다.
for item in plan["review_items"][:5]:
    print("\n주제:", item["title"], "| 원문 ID:", item["record_id"])
    print("PDF 쪽:", item["source_pages"], "| 매뉴얼 시작 쪽:", item["manual_page_number"])
    print("검토 이유:", item["review_reasons"])
    print("청크 수:", item["chunks"], "| 그림 연결 수:", item["image_links"])


## 4. 실제 조사 결과와 다음 저장 범위 제안

2026-10-02 Python 실행 결과: 부모 527개·청크 2,213개, 고유 그림 864개·주제 연결 1,157건. 설명 있는 연결 10건, 없는 연결 1,147건입니다. 자동 초안은 507개이고 현재 개인 DB는 23행·512KiB 그대로입니다.

조사 당시 추가 발견: 03번 노트북의 PDF 74쪽 I2.jpg·I7.jpg 설명이 Python에 반영되지 않았습니다. 이후 사용자 승인으로 두 설명과 관련 제목을 반영했습니다. 현재 설명 있는 연결 12건·없는 연결 1,145건이며 보완·전체 행 준비는 20번 노트북에서 확인합니다.

1. 전체 글·벡터·그림 참조를 **새 처리 작업(run_id)** 으로 개인 스키마에 저장합니다. 기존 표본 작업은 보존합니다.
2. 같은 PDF 정보와 이미 등록한 그림 3개는 재사용하도록 저장 서비스를 준비합니다.
3. 자동 초안·그림 연결의 검토 상태와 각주 연결을 그대로 보존합니다. 저장했다고 검토 완료로 바꾸지 않습니다.
4. 이번 전체 글 저장에서는 추가 그림 파일 업로드를 0개로 두고, 그림은 관련 주제별 설명·배치를 확인한 뒤 단계적으로 업로드합니다.
5. 이후 전체 저장 서비스 코드는 준비했으며 실제 전체 임베딩·저장은 다음 작업입니다. 이 조사 함수의 `ready_for_full_save=False`는 실제 벡터·저장이 준비되지 않은 상태를 뜻합니다.

이 제안은 최종 그림 검색 목표를 유지하면서 글 검색과 파일 업로드 범위를 나누는 방식입니다. M5의 근거 부족 안내와 답변 구성은 아직 구현 전입니다.

**Python 코드 읽기:** `manual.py storage-plan` → `storage_plan.py build_storage_plan` → `manual_service.py prepare`.
